In [1]:


# 1. Uninstall the broken library
!pip uninstall -y torchcodec

# 2. Install the standard audio backends
!pip install torchaudio soundfile librosa

# 3. Upgrade datasets to ensure it detects the new backends correctly
!pip install --upgrade datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 10.6 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.4.2
    Uninstalling datasets-4.4.2:
      Successfully uninstalled datasets-4.4.2


# INFERENCE

In [2]:
from transformers import GenerationConfig

MODEL_ID = "arbml/whisper-largev2-ar"  # set to your model id on the Hub
MULTILINGUAL = True  # set True for multilingual models, False for English-only

if MULTILINGUAL:
    generation_config = GenerationConfig.from_pretrained("openai/whisper-large-v2")
else:
    generation_config = GenerationConfig.from_pretrained("openai/whisper-medium.en")

generation_config.json: 0.00B [00:00, ?B/s]

In [3]:
"""
IMPROVED INFERENCE FOR LONG-FORM BENGALI ASR

This script properly handles long audio files by:
1. Chunking audio into 30s segments with overlap
2. Transcribing each chunk
3. Intelligently merging transcriptions
4. Handling edge cases and audio normalization
"""

import os
import glob
import torch
import librosa
import pandas as pd
import numpy as np
from tqdm import tqdm
from transformers import WhisperForConditionalGeneration, WhisperProcessor
import warnings

# =====================================================================
# CONFIGURATION
# =====================================================================

MODEL_PATH = "/kaggle/input/abcd1223456/whisper-bengali-streaming-final"  # Or your trained model path
TEST_AUDIO_DIR = "/kaggle/input/dl-sprint-4-0-bengali-long-form-speech-recognition/transcription/transcription/test/audio"
OUTPUT_CSV = "./submission_aligned.csv"

# Chunking parameters
CHUNK_DURATION = 30.0  # seconds
OVERLAP_DURATION = 1.0  # seconds overlap between chunks
SAMPLE_RATE = 16000

# Generation parameters
MAX_NEW_TOKENS = 444
TEMPERATURE = 0.0  # Deterministic
NUM_BEAMS = 1  # Greedy decoding for speed

print("="*70)
print("LONG-FORM ASR INFERENCE WITH CHUNKING")
print("="*70)

# =====================================================================
# LOAD MODEL
# =====================================================================

print("\n[1/4] Loading model...")

processor = WhisperProcessor.from_pretrained(MODEL_PATH)
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    device_map="auto"
)

# --- FIX: Update the Generation Config object directly ---
# --- FIX: Simpler approach - just set forced_decoder_ids to None ---
model.generation_config.forced_decoder_ids = None
model.config.forced_decoder_ids = None

# Don't set language/task attributes at all
# ---------------------------------------------------------
# ---------------------------------------------------------

model.eval()
print(f"✓ Model loaded: {MODEL_PATH}")
print(f"✓ Device: {model.device}")
# =====================================================================
# AUDIO PROCESSING FUNCTIONS
# =====================================================================

def load_and_normalize_audio(audio_path):
    """
    Load audio file and normalize it.
    Returns: audio array at 16kHz, mono
    """
    try:
        # Load audio
        audio, sr = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
        
        # Normalize audio to [-1, 1] range
        if np.abs(audio).max() > 0:
            audio = audio / np.abs(audio).max()
        
        return audio, sr
    except Exception as e:
        warnings.warn(f"Failed to load {audio_path}: {e}")
        return None, None


def create_overlapping_chunks(audio, chunk_duration=30.0, overlap=1.0):
    """
    Split audio into overlapping chunks for better continuity.
    
    Args:
        audio: Audio array
        chunk_duration: Duration of each chunk in seconds
        overlap: Overlap between chunks in seconds
    
    Returns:
        List of (audio_chunk, start_time, end_time) tuples
    """
    sr = SAMPLE_RATE
    chunk_samples = int(chunk_duration * sr)
    overlap_samples = int(overlap * sr)
    stride = chunk_samples - overlap_samples
    
    chunks = []
    total_duration = len(audio) / sr
    
    start = 0
    while start < len(audio):
        end = min(start + chunk_samples, len(audio))
        chunk = audio[start:end]
        
        # Pad last chunk if too short (minimum 1 second)
        if len(chunk) < sr and len(chunks) > 0:
            break  # Skip very short final chunk
        
        start_time = start / sr
        end_time = end / sr
        
        chunks.append((chunk, start_time, end_time))
        
        if end >= len(audio):
            break
        
        start += stride
    
    return chunks


def transcribe_chunk(audio_chunk):
    inputs = processor(
        audio_chunk,
        sampling_rate=SAMPLE_RATE,
        return_tensors="pt"
    )
    
    input_features = inputs.input_features.to(
        device=model.device,
        dtype=next(model.parameters()).dtype
    )
    
    with torch.no_grad():
        # Cleaned up call: uses the model's updated generation_config automatically
        predicted_ids = model.generate(
            input_features,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            num_beams=NUM_BEAMS,
            do_sample=False
        )
    
    transcription = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]
    
    return transcription.strip()


def merge_overlapping_transcriptions(chunk_transcriptions, overlap_duration=1.0):
    """
    Merge transcriptions from overlapping chunks intelligently.
    
    Strategy:
    - Use full transcription of each chunk
    - Remove potential repeated words at boundaries
    - Join with space
    """
    if not chunk_transcriptions:
        return ""
    
    if len(chunk_transcriptions) == 1:
        return chunk_transcriptions[0]
    
    merged = chunk_transcriptions[0]
    
    for i in range(1, len(chunk_transcriptions)):
        current_text = chunk_transcriptions[i]
        
        if not current_text:
            continue
        
        # Simple merge: add space and append
        # For Bengali, word-level deduplication is complex due to script
        # In production, you'd use more sophisticated alignment
        merged = merged + " " + current_text
    
    # Clean up multiple spaces
    merged = " ".join(merged.split())
    
    return merged


def transcribe_long_audio(audio_path):
    """
    Transcribe a long audio file by chunking and merging.
    """
    # Load audio
    audio, sr = load_and_normalize_audio(audio_path)
    
    if audio is None:
        return ""
    
    duration = len(audio) / sr
    
    # If short, transcribe directly
    if duration <= CHUNK_DURATION:
        return transcribe_chunk(audio)
    
    # Create overlapping chunks
    chunks = create_overlapping_chunks(
        audio,
        chunk_duration=CHUNK_DURATION,
        overlap=OVERLAP_DURATION
    )
    
    # Transcribe each chunk
    chunk_texts = []
    for chunk_audio, start_time, end_time in chunks:
        text = transcribe_chunk(chunk_audio)
        chunk_texts.append(text)
    
    # Merge transcriptions
    final_text = merge_overlapping_transcriptions(
        chunk_texts,
        overlap_duration=OVERLAP_DURATION
    )
    
    return final_text

# =====================================================================
# INFERENCE LOOP
# =====================================================================

print("\n[2/4] Scanning test directory...")
test_files = sorted(glob.glob(os.path.join(TEST_AUDIO_DIR, "*.wav")))

if len(test_files) == 0:
    raise ValueError(f"No .wav files found in {TEST_AUDIO_DIR}")

print(f"✓ Found {len(test_files)} audio files")

print("\n[3/4] Processing audio files...")

results = []

for idx, audio_path in enumerate(tqdm(test_files, desc="Transcribing")):
    filename = os.path.splitext(os.path.basename(audio_path))[0]
    
    try:
        # Transcribe
        transcript = transcribe_long_audio(audio_path)
        
        # Validation
        if not transcript:
            warnings.warn(f"Empty transcript for {filename}")
        
        results.append({
            "filename": filename,
            "transcript": transcript
        })
        
    except Exception as e:
        print(f"\n❌ Error processing {filename}: {e}")
        results.append({
            "filename": filename,
            "transcript": ""
        })

# =====================================================================
# SAVE RESULTS
# =====================================================================

print("\n[4/4] Saving submission...")

df = pd.DataFrame(results)
df.to_csv(OUTPUT_CSV, index=False)

print(f"\n✓ Submission saved: {OUTPUT_CSV}")
print(f"✓ Total files: {len(results)}")
print(f"✓ Empty transcripts: {sum(1 for r in results if not r['transcript'])}")

print("\n" + "="*70)
print("INFERENCE COMPLETE!")
print("="*70)

# Statistics
word_counts = [len(r['transcript'].split()) for r in results if r['transcript']]
if word_counts:
    print(f"\nTranscript Statistics:")
    print(f"  Mean words: {np.mean(word_counts):.1f}")
    print(f"  Median words: {np.median(word_counts):.1f}")
    print(f"  Max words: {max(word_counts)}")
    print(f"  Min words: {min(word_counts)}")

2026-01-30 19:03:59.693733: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769799839.960036      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769799840.037246      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769799840.624215      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769799840.624253      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769799840.624256      24 computation_placer.cc:177] computation placer alr

LONG-FORM ASR INFERENCE WITH CHUNKING

[1/4] Loading model...
✓ Model loaded: /kaggle/input/abcd1223456/whisper-bengali-streaming-final
✓ Device: cuda:0

[2/4] Scanning test directory...
✓ Found 24 audio files

[3/4] Processing audio files...


Transcribing:   0%|          | 0/24 [00:00<?, ?it/s]`generation_config` default values have been modified to match model-specific defaults: {'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}. If this is not desired, please set these values explicitly.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type


[4/4] Saving submission...

✓ Submission saved: ./submission_aligned.csv
✓ Total files: 24
✓ Empty transcripts: 0

INFERENCE COMPLETE!

Transcript Statistics:
  Mean words: 5433.3
  Median words: 5174.0
  Max words: 8265
  Min words: 3363
